In [ ]:
import pickle
import re
import string

from google.colab import drive, files
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from wordcloud import WordCloud
import xgboost as xgb

In [ ]:
nltk.download('punkt')  # For word_tokenize
nltk.download('stopwords')  # For stopwords
nltk.download('wordnet')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
import os

def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

real_path = find_file('Reviews.csv', '/content/drive/MyDrive')

if real_path:
    print(f" {real_path}")
else:
    print("not found")
df = pd.read_csv(real_path)

df.head()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:

ax = df['Score'].value_counts().sort_index() \
    .plot(kind='bar',
          title='Count of Reviews by Stars',
          figsize=(10, 5))

ax.set_xlabel('Review Stars')
plt.show()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df_copy = df.copy()

df = df[['Text', 'Summary', 'Score']]
df = df.drop_duplicates(subset=['Text'], keep='first')
df.head()

In [ ]:
df.duplicated().sum()

In [ ]:
def cleaning(text):



    text = re.sub(r'https?:\/\/\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)
    text = re.sub(r'#[A-Za-z0-9_]+', ' ', text)
    text=re.sub(r'<.*?>',' ',text)
    text=text.lower()
    text = text.split()
    text = ' '.join(text)
    return text



In [ ]:
df['cleaning']=df['Text'].apply(cleaning)

In [ ]:
import string
punc=string.punctuation
def remove_punc(text):
    return text.translate(str.maketrans('','',punc))


In [ ]:
df['Remove_punc']=df['cleaning'].apply(remove_punc)

In [ ]:
df.head()

In [ ]:
nltk.download('punkt_tab')

In [ ]:
### word tokenization
from nltk.tokenize import word_tokenize
df['tokenized']=df['cleaning'].apply(word_tokenize)

In [ ]:
df.head()

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

important_words = {
    'not', 'no', 'nor', 'never',
    'very', 'too', 'so',
    'but', 'however'
}

custom_stopwords = stop_words - important_words

def remove_stopWords(words):
    return [word for word in words if word.lower() not in custom_stopwords]

In [ ]:
df['Nostopword_text']=df['tokenized'].apply(remove_stopWords)

In [ ]:
df.head()

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
def lemmatizer_word(text):
 lemmas=[lemmatizer.lemmatize(word,pos='v') for word in text  ]
 return lemmas
df['lemmatizer_text']=df['Nostopword_text'].apply(lemmatizer_word)


In [ ]:
df.head()

In [ ]:
df['clean_text'] = df['lemmatizer_text'].apply(lambda tokens: " ".join(tokens))


In [ ]:
for text in df['Text'][3000:3005]:
    print(text)
    print('-' * 40)

In [ ]:
for text in df['clean_text'][3000:3005]:
    print(text)
    print('-' * 40)

In [ ]:
text = " ".join(df['clean_text'])

In [ ]:
text

In [ ]:
# wordcloud = WordCloud(width=800, height=400).generate(text)

# plt.imshow(wordcloud)
# plt.axis('off')
# plt.title("Overall Word Cloud")
# plt.show()

In [ ]:
# df.drop(['cleaning','Remove_punc','tokenized','Nostopword_text','lemmatizer_text'],axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
import pandas as pd

required_columns = ['Time', 'Score', 'Text', 'HelpfulnessNumerator', 'HelpfulnessDenominator']
if all(col in df.columns for col in required_columns):

    df['Time'] = pd.to_datetime(df['Time'], unit='s')
    df['ReviewYear'] = df['Time'].dt.year
    df['ReviewMonth'] = df['Time'].dt.month
    df['DayOfWeek'] = df['Time'].dt.dayofweek
else:
    print("Missing columns! Please check: ", df.columns)

In [ ]:
import string

df['text_length'] = df['Text'].str.len()
df['word_count'] = df['Text'].apply(lambda x: len(str(x).split()))
df['avg_word_length'] = df['text_length'] / (df['word_count'] + 1)
df['avg_sentence_length'] = df['text_length'] / df['Text'].apply(lambda x: len(str(x).split('.')))

In [ ]:
df['punct_count'] = df['Text'].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))
df['punct_percent'] = df['punct_count'] / (df['text_length'] + 1)
df['caps_ratio'] = df['Text'].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / (len(str(x)) + 1))

In [ ]:
df['exclamation_count'] = df['Text'].str.count('!')
df['question_count'] = df['Text'].str.count('\?')

In [ ]:
# from textblob import TextBlob

# sentiments = [TextBlob(str(text)).sentiment for text in df['Text']]

# df['polarity'] = [s.polarity for s in sentiments]
# df['subjectivity'] = [s.subjectivity for s in sentiments]

In [ ]:
df['lexical_richness'] = df['Text'].apply(lambda x: len(set(str(x).split())) / len(str(x).split()) if len(str(x).split()) > 0 else 0)
df['summary_length'] = df['Summary'].str.len()
df['summary_ratio'] = df['summary_length'] / (df['text_length'] + 1)

In [ ]:
cols_to_show = ['text_length', 'avg_sentence_length', 'caps_ratio', 'lexical_richness']
print(df[cols_to_show].head())

In [ ]:
# from gensim.models.fasttext import load_facebook_model
# import numpy as np
# import pandas as pd

# model = load_facebook_model('cc.en.300.bin')

# def get_fasttext_features(text):
#     words = str(text).lower().split()
#     vectors = [model.wv[word] for word in words if word in model.wv]
#     if not vectors:
#         return np.zeros(model.vector_size)
#     return np.mean(vectors, axis=0)

# fasttext_vectors = df['Text'].apply(get_fasttext_features)
# ft_matrix = np.vstack(fasttext_vectors.values)
# ft_df = pd.DataFrame(ft_matrix, index=df.index).add_prefix('ft_')

# df_final = pd.concat([
#     df[['lexical_richness', 'summary_ratio', 'polarity']],
#     ft_df
# ], axis=1)

In [ ]:
df.sample()

In [ ]:
extra_cols = ['lexical_richness', 'summary_ratio', 'text_length', 'caps_ratio']

# FIX: Use 'clean_text' (which is definitively a string) and split it into lists.
# This prevents the data-wipe issue.
X_text = df['clean_text'].apply(lambda x: str(x).split())
X_extra = df[extra_cols]
y = df['Score']

# Train-Test Split
X_text_train, X_text_test, X_extra_train, X_extra_test, y_train, y_test = train_test_split(
    X_text, X_extra, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
!pip install fasttext

In [ ]:
!pip install gensim

In [ ]:

from gensim.models import FastText
from gensim.models.fasttext import load_facebook_model

import multiprocessing
from gensim.models import FastText


In [ ]:
clean_sentences_train = X_text_train.tolist()
clean_sentences_train = [doc for doc in clean_sentences_train if len(doc) > 0]

# Train FastText
cores = multiprocessing.cpu_count()
print(f"Training FastText on {len(clean_sentences_train)} valid samples using {cores} cores...")

model = FastText(sentences=clean_sentences_train, vector_size=100, window=5, min_count=2, workers=cores)
print("FastText model successfully trained!")

model.wv.fill_norms(force=True)

# Vectorization function
def vectorize_text(text_series, wv):
    vectors = []
    for text in text_series:
        if isinstance(text, list) and len(text) > 0:
            vectors.append(wv.get_mean_vector(text))
        else:
            vectors.append(np.zeros(wv.vector_size))
    return np.array(vectors)

# Apply vectorization
X_ft_train = vectorize_text(X_text_train, model.wv)
X_ft_test  = vectorize_text(X_text_test, model.wv)

# Combine FastText features with your extra numerical features
X_train_final = np.hstack([X_ft_train, X_extra_train.values])
X_test_final  = np.hstack([X_ft_test, X_extra_test.values])

In [ ]:
import os
import zipfile
from google.colab import files

model_name = "fasttext_amazon.model"
model.save(model_name)
print("Model saved locally.")

zip_filename = 'fasttext_model.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in os.listdir():
        if file.startswith(model_name):
            zipf.write(file)
            print(f"Added {file} to zip.")

# 3. اعمل داونلود لملف الـ Zip
files.download(zip_filename)
print("Download initiated!")

In [ ]:
scaler = StandardScaler()
X_extra_train_scaled = scaler.fit_transform(X_extra_train)
X_extra_test_scaled  = scaler.transform(X_extra_test)

X_train_final = np.hstack([X_ft_train, X_extra_train_scaled])
X_test_final  = np.hstack([X_ft_test, X_extra_test_scaled])

lr_model = LogisticRegression(
    max_iter=2000,
    multi_class='multinomial',
    solver='lbfgs',
    class_weight='balanced'
)

print("Training Logistic Regression with balanced weights...")
lr_model.fit(X_train_final, y_train)

y_pred = lr_model.predict(X_test_final)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
with open('logistic_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# model.save('my_ft.model')
files.download('logistic_model.pkl')
files.download('scaler.pkl')
# files.download('my_ft.model')

print("Donnnnne")

In [ ]:
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method='hist'
)

print("Applying SMOTE to balance the dataset...")
smote = SMOTE(random_state=42)

# Resample the training data using the FINAL combined array
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_final, y_train_xgb)

print("Training XGBoost on balanced data...")

xgb_model.fit(X_train_balanced, y_train_balanced)

y_pred_xgb_raw = xgb_model.predict(X_test_final)

y_pred_final = y_pred_xgb_raw + 1

print(f"\nXGBoost Accuracy: {accuracy_score(y_test, y_pred_final):.2f}")
print("\nXGBoost Classification Report:\n", classification_report(y_test, y_pred_final))

with open('xgboost_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

print("XGBoost model saved. You can predict by adding 1 to the model output.")

files.download('xgboost_model.pkl')
print("Download initiated.")